# Step 2: 도구(Tool) 정의 시스템 — 타입 안전한 인터페이스

## 📋 학습 목표

- Claude Code가 ~40개의 도구를 **하나의 통일된 인터페이스**로 관리하는 방법을 이해합니다
- **buildTool() 팩토리**의 "fail-closed 기본값" 설계 철학을 배웁니다
- Python으로 **Tool 프로토콜**, **ToolRegistry**, 그리고 실제 도구 3개를 구현합니다

> Step 1에서 도구를 딕셔너리와 if/elif로 정의했습니다. 도구가 3개일 때는 괜찮지만,
> 40개가 되면? Claude Code는 이 문제를 **도구 인터페이스**로 해결합니다.

---

## 🔍 Claude Code 소스 분석

### 2-1. Tool 타입 — 모든 도구가 따르는 계약 (Tool.ts:362-410)

Claude Code의 모든 도구(Bash, FileRead, FileEdit, WebSearch 등)는 `Tool` 타입을 구현합니다:

```typescript
// src/Tool.ts:362-410 (핵심 필드만 발췌)

export type Tool<Input, Output> = {
  name: string                    // 도구 이름 — LLM이 이 이름으로 호출
  aliases?: string[]              // 대체 이름 (이전 버전 호환)

  // ★ 핵심 4가지
  readonly inputSchema: Input             // Zod 스키마로 입력 검증
  call(args, context, ...): Promise<ToolResult<Output>>  // 실행 로직
  description(input, options): Promise<string>            // LLM에게 보여줄 설명
  checkPermissions(input, ctx): Promise<PermissionResult> // 권한 확인

  // ★ 실행 특성 — 오케스트레이션에서 사용 (Step 3)
  isReadOnly(input): boolean           // 읽기 전용? → 병렬 실행 가능 여부에 영향
  isConcurrencySafe(input): boolean    // 병렬 실행 안전?
  isDestructive?(input): boolean       // 파괴적 작업? (삭제, 덮어쓰기)
  isEnabled(): boolean                 // 활성화 상태?

  // ★ 입력 검증
  validateInput?(input, context): Promise<ValidationResult>

  // UI 렌더링 (React/Ink)
  renderToolUseMessage(input, options): React.ReactNode
  renderToolResultMessage(output, ...): React.ReactNode

  maxResultSizeChars: number           // 결과 최대 크기 (기본 30,000자)
}
```

**핵심 포인트**: 모든 도구가 같은 인터페이스를 따르기 때문에, 에이전트 루프(`queryLoop`)는 BashTool인지 FileReadTool인지 모릅니다. 그냥 `tool.call(args)`만 호출하면 됩니다.

### 2-2. buildTool()과 TOOL_DEFAULTS — 안전한 기본값 (Tool.ts:757-792)

```typescript
// src/Tool.ts:757-769

const TOOL_DEFAULTS = {
  isEnabled: () => true,
  isConcurrencySafe: () => false,    // ★ 기본: 병렬 불안전 (더 제한적)
  isReadOnly: () => false,           // ★ 기본: 쓰기 가능 (더 제한적)
  isDestructive: () => false,
  checkPermissions: (input) =>
    Promise.resolve({ behavior: 'allow', updatedInput: input }),
  // ...
}

// Tool.ts:783-792
export function buildTool<D>(def: D): BuiltTool<D> {
  return {
    ...TOOL_DEFAULTS,      // 1. 안전한 기본값
    userFacingName: () => def.name,
    ...def,                // 2. 사용자 정의가 기본값을 덮어씀
  }
}
```

**"Fail-closed" 설계 철학**: 기본값이 항상 더 제한적인 쪽입니다.
- `isConcurrencySafe = false` → 명시적으로 안전하다고 선언하지 않으면 직렬 실행
- `isReadOnly = false` → 명시적으로 읽기전용이라고 선언하지 않으면 쓰기 가능으로 취급

### 2-3. 도구 레지스트리 — getAllBaseTools() (tools.ts:193-249)

```typescript
// src/tools.ts:193-249 (핵심만 발췌)

export function getAllBaseTools(): Tools {
  return [
    AgentTool,
    BashTool,
    GlobTool, GrepTool,
    FileReadTool, FileEditTool, FileWriteTool,
    WebFetchTool, WebSearchTool,
    SkillTool,
    // ... 환경/피처 플래그에 따라 조건부 포함
    ...(isWorktreeModeEnabled() ? [EnterWorktreeTool, ExitWorktreeTool] : []),
    ...(SleepTool ? [SleepTool] : []),
  ]
}
```

도구 등록은 단순한 배열입니다. 조건부 포함(`...spread`)으로 환경에 따라 도구를 추가/제거합니다.

---

## 🏗️ Python으로 구현하기

3개 모듈을 만듭니다:
1. `tool_base.py` — Tool ABC + build_tool() 팩토리
2. `tool_registry.py` — 도구 레지스트리
3. `tools/bash_tool.py`, `file_read_tool.py`, `file_write_tool.py` — 구체적 도구 3개

### 구현 확인: Tool ABC와 build_tool()

In [ ]:
import sys, os
sys.path.insert(0, ".")

# build_tool()로 간단한 도구를 만들어봅시다
from mini_claude.tool_base import build_tool, ToolResult, Tool

# 예시: 현재 시간 도구 (Step 1의 딕셔너리 방식 vs build_tool 방식 비교)
import datetime

async def _get_time(args: dict) -> ToolResult:
    tz = args.get("timezone", "UTC")
    now = datetime.datetime.now(datetime.timezone.utc).isoformat()
    return ToolResult(data=f"현재 시간 ({tz}): {now}")

time_tool = build_tool(
    name="get_current_time",
    description="현재 시간을 반환합니다",
    input_schema={
        "type": "object",
        "properties": {
            "timezone": {"type": "string", "description": "시간대. 기본: UTC"}
        },
    },
    call=_get_time,
    read_only=True,           # 상태를 변경하지 않음
    concurrency_safe=True,    # 동시 호출 안전
)

# Tool 인터페이스 확인
print(f"이름: {time_tool.name}")
print(f"설명: {time_tool.description}")
print(f"읽기전용: {time_tool.is_read_only()}")
print(f"병렬 안전: {time_tool.is_concurrency_safe()}")
print(f"파괴적: {time_tool.is_destructive()}")
print(f"isinstance(Tool): {isinstance(time_tool, Tool)}")

### 실제 도구 3개: Bash, FileRead, FileWrite

`mini_claude/tools/` 디렉토리에 구현된 3개의 도구를 확인합니다.

In [ ]:
from mini_claude.tools.bash_tool import BashTool
from mini_claude.tools.file_read_tool import FileReadTool
from mini_claude.tools.file_write_tool import FileWriteTool

# 각 도구의 특성 비교
tools = [BashTool, FileReadTool, FileWriteTool]

print(f"{'도구':<12} {'읽기전용':<10} {'병렬안전':<10} {'파괴적':<8}")
print("─" * 42)
for t in tools:
    print(f"{t.name:<12} {str(t.is_read_only()):<10} {str(t.is_concurrency_safe()):<10} {str(t.is_destructive()):<8}")

# 도구 실행 테스트
result = await FileReadTool.call({"file_path": os.path.abspath("README.md"), "limit": 5})
print(f"\n📄 FileRead 테스트 (처음 5줄):\n{result.content[:300]}")

### ToolRegistry — 도구를 모아서 관리

`ToolRegistry`는 도구를 등록하고, 이름으로 찾고, API 스키마를 자동 생성합니다.

In [ ]:
from mini_claude.tool_registry import ToolRegistry

# 레지스트리 생성 및 도구 등록
registry = ToolRegistry()
registry.register(BashTool)
registry.register(FileReadTool)
registry.register(FileWriteTool)

print(f"등록된 도구: {registry.get_names()}")
print(f"총 {len(registry)}개")

# 이름으로 도구 찾기 — Claude Code의 findToolByName() (Tool.ts:358)
tool = registry.find_by_name("Read")
print(f"\n'Read' 찾기: {tool.name if tool else 'Not found'}")

# API 스키마 자동 생성
schemas = registry.to_anthropic_schemas()
print(f"\nAnthropic 스키마 (첫 번째):")
import json
print(json.dumps(schemas[0], indent=2, ensure_ascii=False)[:300])

---

## 🧪 실습: 도구 시스템을 에이전트 루프에 통합

Step 1의 에이전트 루프에 ToolRegistry를 연결합니다.
`registry.execute()`를 `tool_executor`로 넘기면, 에이전트 루프가 자동으로 등록된 도구를 실행합니다.

In [ ]:
from mini_claude.llm_client import create_client, AnthropicClient
from mini_claude.agent_loop import run_agent

client = create_client()

# 클라이언트 타입에 맞는 스키마 선택
provider = "anthropic" if isinstance(client, AnthropicClient) else "openai"
tools_schema = registry.to_api_schemas(provider)

# ★ registry.execute를 tool_executor로 연결
# 이것이 도구 시스템과 에이전트 루프를 연결하는 다리입니다
result = await run_agent(
    client=client,
    prompt="현재 디렉토리의 파일 목록을 보여주고, README.md의 첫 10줄을 읽어줘.",
    system="당신은 도구를 사용하여 파일 시스템을 탐색하는 AI 어시스턴트입니다. 한국어로 답변하세요.",
    tools=tools_schema,
    tool_executor=registry.execute,  # ← ToolRegistry가 도구 실행을 담당
    verbose=True,
)

print("\n" + "=" * 60)
print("📝 최종 응답:")
print(result[:500])

---

## 💡 핵심 설계 원칙 정리

### 1. 통일된 인터페이스 (Tool ABC)
모든 도구가 `name`, `input_schema`, `call()`, `description`을 구현합니다. 에이전트 루프는 구체적 도구를 몰라도 됩니다. 새 도구를 추가할 때 루프를 수정할 필요가 없습니다.

### 2. Fail-closed 기본값
`buildTool()`의 기본값은 항상 더 제한적인 쪽입니다. "안전하다고 명시적으로 선언하지 않으면 위험하다고 가정"하는 원칙입니다. 이를 통해 새 도구 작성 시 실수로 위험한 기본 동작이 되는 것을 방지합니다.

### 3. 레지스트리 패턴
도구를 `ToolRegistry`에 모아두면: 이름으로 조회, API 스키마 자동 생성, 일괄 실행이 가능합니다. `registry.execute()`를 에이전트 루프의 `tool_executor`로 넘기는 것만으로 전체 시스템이 연결됩니다.

---

## ➡️ 다음 Step 미리보기

현재 도구는 순차 실행됩니다. 하지만 `FileReadTool`은 `is_concurrency_safe=True`이므로 여러 파일을 동시에 읽을 수 있습니다.

**Step 3**에서는 Claude Code의 `toolOrchestration.ts`를 분석하여:
- `is_read_only()`와 `is_concurrency_safe()`에 따라 **병렬/직렬 실행을 자동 결정**하는 오케스트레이터를 구현합니다.